# 06 · Reshape repeated wages from wide to long

**Module:** 2 · **Time:** 45 minutes · **Prerequisites:** Merging and identifying keys

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/06_reshaping.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F06_reshaping.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is private and badges point to `main` after merge.

## Learning goals

- **T**ransform two yearly wage columns into a long person-wave table.
- **U**se a unique record key for reversible reshaping.
- **P**ivot long data back to wide and verify row counts.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Fictional observations:", len(survey))

## Why this matters

Repeated outcomes are often stored with years in column names. Many panel and time-series tasks need one observation per person-year instead. The fictional dataset includes duplicate `person_id` values on purpose, so we first create a unique row record key before reshaping.

## Create a unique record identifier

A name that looks like a person ID is not necessarily unique. Here a row identifier preserves each originally distinct record, without falsely resolving the duplicate person ID.

In [ ]:
wide = survey.reset_index(names="record_id").copy()
wide["record_id"] = wide["record_id"] + 1
assert wide["record_id"].is_unique
print(wide[["record_id", "person_id"]].head())

## Melt to one row per record-year

`melt` keeps identifier columns and gathers the two annual wage columns into the variables `wave` and `wage_text`.

In [ ]:
long = wide.melt(
    id_vars=["record_id", "person_id", "household_id"],
    value_vars=["wage_2023", "wage_2024"],
    var_name="wave", value_name="wage_text"
)
long["year"] = long["wave"].str[-4:].astype(int)
print(long.head(3))

## Pivot back to wide

`pivot` requires one value per `record_id` and year. The original `person_id` is repeated, so pivoting on it alone would be ambiguous.

In [ ]:
reconstructed = long.pivot(index="record_id", columns="year", values="wage_text")
assert len(reconstructed) == len(wide)
print(reconstructed.head())
print("Recovered year columns:", reconstructed.columns.tolist())

## A clean panel needs defined units

Before calculating within-person changes, verify whether two records actually refer to the same person. A unique record ID is not a verified longitudinal person identifier.

In [ ]:
duplicates = wide["person_id"].duplicated(keep=False)
print("Respondent IDs needing review:", wide.loc[duplicates, "person_id"].tolist())
print("Panel row count:", len(long))
assert len(long) == 2 * len(wide)

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Wide to long | `melt(id_vars=..., value_vars=...)` | `reshape long wage_, i(id) j(year)` | `pivot_longer(...)` |
| Long to wide | `pivot(index=..., columns=..., values=...)` | `reshape wide wage_, i(id) j(year)` | `pivot_wider(...)` |
| Unique key check | `df[key].is_unique` | `isid key` | `dplyr::n_distinct(key) == n()` |

Reshaping requires a clearly defined key; a duplicated person ID is a research-data problem, not a formatting inconvenience.

## Try it

Complete the answer cells, then run the friendly checks. Blank learner exercises produce hints rather than exceptions.

### ★ Exercise 1 — Check the new key

Store `record_key_unique` as a Boolean indicating whether `wide['record_id']` is unique.

In [ ]:
record_key_unique = None

In [ ]:
if isinstance(record_key_unique, (bool, np.bool_)) and record_key_unique:
    print("✓ Record key is unique")
else:
    print("Hint: use Series.is_unique to inspect the row key.")

### ★★ Exercise 2 — Create a long wage dataset

Make `long_wages` by melting `wide` using `record_id`, and two year-specific wage columns.

In [ ]:
long_wages = None

In [ ]:
expected = wide.melt(id_vars=["record_id"],
                     value_vars=["wage_2023", "wage_2024"],
                     var_name="wave", value_name="wage_text")
if isinstance(long_wages, pd.DataFrame) and long_wages.equals(expected):
    print("✓ Wide to long conversion correct")
else:
    print("Hint: melt and name the year and wage columns explicitly.")

### ★★ Exercise 3 — Extract the survey year

Create `wave_year` as integer years from `long['wave']` using the last four characters.

In [ ]:
wave_year = None

In [ ]:
expected = long["wave"].str[-4:].astype(int)
if isinstance(wave_year, pd.Series) and wave_year.equals(expected):
    print("✓ Year parsed correctly")
else:
    print("Hint: use the .str accessor to take the final four characters.")

### ★★★ Exercise 4 — Pivot back and compare

Create `wide_rebuilt` from `long` with one row per `record_id`, year columns, and wage text values.

In [ ]:
wide_rebuilt = None

In [ ]:
expected = long.pivot(index="record_id", columns="year", values="wage_text")
if isinstance(wide_rebuilt, pd.DataFrame) and wide_rebuilt.equals(expected):
    print("✓ Long to wide conversion correct")
else:
    print("Hint: pivot with record_id, year, and wage_text.")

## Common errors

- `ValueError: Index contains duplicate entries, cannot reshape` occurs when a pivot key does not identify rows uniquely.
- Mixing a longitudinal person ID with an observation row ID can produce invalid panel interpretations.
- The `melt` result repeats identifier values by design; check expected row counts.
- The year may be text after melting; parse it before time-series arithmetic.

## Recap

- Wide data store years across columns; long data store years as values.
- Use a verified unique key to reshape without losing observations.
- `melt` gathers years and `pivot` spreads them.
- Duplicate person identifiers require investigation before claiming panel structure.

## Next steps

Next: [Groupby and aggregation](07_groupby_aggregate.ipynb) (planned). Reference: [pandas reshape user guide](https://pandas.pydata.org/docs/user_guide/reshaping.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).